In [1]:
import pandas as pd

from pylondrina.importing import import_trips_from_dataframe
from pylondrina.schema import FieldSpec, TripSchema
from pylondrina.errors import ImportError as PylondrinaImportError

SOFT_CAP = 256
HARD_CAP = 1024

### Test 1 - Import normal de columnas

En esta prueba se construye un `TripSchema` pequeño y un `DataFrame` angosto, de modo que el import produzca un `TripDataset` normal. El objetivo es verificar que, en un caso sin ancho problemático, no aparezcan los codes nuevos de control de columnas: `IMP.COLUMNS.WIDE_TABLE` ni `IMP.COLUMNS.HARD_CAP_EXCEEDED`.

In [ ]:
schema_normal = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_note": FieldSpec(name="user_note", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
        "mode_hint": FieldSpec(name="mode_hint", dtype="string", required=False),
    },
    required=["movement_id", "user_note"],
)

df_normal = pd.DataFrame(
    {
        "movement_id": ["m0", "m1"],
        "user_note": ["fila_a", "fila_b"],
        "origin_time_utc": ["2024-01-01T08:00:00Z", "2024-01-01T09:00:00Z"],
        "destination_time_utc": ["2024-01-01T08:10:00Z", "2024-01-01T09:10:00Z"],
        "mode_hint": ["bus", "metro"],
    }
)

trips_normal, report_normal = import_trips_from_dataframe(
    df=df_normal,
    schema=schema_normal,
)

cap_issue_codes_normal = [
    issue.code
    for issue in report_normal.issues
    if issue.code.startswith("IMP.COLUMNS.")
]

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_normal.data)
display(report_normal.issues)

print("\nColumnas finales:", len(trips_normal.data.columns))
print("Codes de cap detectados:", cap_issue_codes_normal)
print("Todos los codes del reporte:", [issue.code for issue in report_normal.issues])

assert len(trips_normal.data.columns) < SOFT_CAP
assert "IMP.COLUMNS.WIDE_TABLE" not in cap_issue_codes_normal
assert "IMP.COLUMNS.HARD_CAP_EXCEEDED" not in cap_issue_codes_normal

print("OK: import normal sin warnings/errors de cap de columnas.")

,movement_id,user_note,origin_time_utc,destination_time_utc,mode_hint
0,m0,fila_a,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,bus
1,m1,fila_b,2024-01-01 09:00:00+00:00,2024-01-01 09:10:00+00:00,metro


[Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_4197cb1f66f74d66b0bd5b130a862356'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_4197cb1f66f74d66b0bd5b130a862356', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]


Columnas finales: 5
Codes de cap detectados: []
Todos los codes del reporte: ['IMP.METADATA.DATASET_ID_CREATED']
OK: import normal sin warnings/errors de cap de columnas.


### Test 2 - Import con soft cap excedido usando solo campos del schema

En esta prueba se construye un `TripSchema` ancho, con más de 256 campos efectivos, pero sin sobrepasar el hard cap. El `DataFrame` contiene únicamente columnas que ya están declaradas en el schema, es decir, no se depende de campos extra o extendidos. El objetivo es verificar que el import continúe, pero emita el warning `IMP.COLUMNS.WIDE_TABLE` y no el error de hard cap.

In [6]:
n_wide_schema_fields = 260

fields_soft_schema = {
    "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
    "user_note": FieldSpec(name="user_note", dtype="string", required=True),
    "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
    "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
}

for i in range(n_wide_schema_fields):
    field_name = f"wide_{i:03d}"
    fields_soft_schema[field_name] = FieldSpec(
        name=field_name,
        dtype="string",
        required=False,
    )

schema_soft_schema_only = TripSchema(
    version="1.0-test",
    fields=fields_soft_schema,
    required=["movement_id", "user_note"],
)

row_soft_schema = {
    "movement_id": "m0",
    "user_note": "ok",
    "origin_time_utc": "2024-01-01T08:00:00Z",
    "destination_time_utc": "2024-01-01T08:10:00Z",
}

for i in range(n_wide_schema_fields):
    row_soft_schema[f"wide_{i:03d}"] = f"valor_{i}"

df_soft_schema = pd.DataFrame([row_soft_schema])

trips_soft_schema, report_soft_schema = import_trips_from_dataframe(
    df=df_soft_schema,
    schema=schema_soft_schema_only,
)

cap_issue_codes_soft_schema = [
    issue.code
    for issue in report_soft_schema.issues
    if issue.code.startswith("IMP.COLUMNS.")
]

wide_issue = next(
    issue
    for issue in report_soft_schema.issues
    if issue.code == "IMP.COLUMNS.WIDE_TABLE"
)

print("Inspección de dataframe importado y de los Issues encontrados")
display(trips_soft_schema.data)
display(report_soft_schema.issues)

print("Columnas finales:", len(trips_soft_schema.data.columns))
print("Codes de cap detectados:", cap_issue_codes_soft_schema)
print("Detalles del warning:", wide_issue.details)
print("extra_fields_kept:", trips_soft_schema.metadata.get("extra_fields_kept"))

assert SOFT_CAP < len(trips_soft_schema.data.columns) < HARD_CAP
assert "IMP.COLUMNS.WIDE_TABLE" in cap_issue_codes_soft_schema
assert "IMP.COLUMNS.HARD_CAP_EXCEEDED" not in cap_issue_codes_soft_schema

# Este caso debe ser schema-only, sin columnas extra conservadas.
assert trips_soft_schema.metadata.get("extra_fields_kept") == []

print("OK: soft cap detectado correctamente con columnas definidas solo en el schema.")

Inspección de dataframe importado y de los Issues encontrados


,movement_id,user_note,origin_time_utc,destination_time_utc,wide_000,wide_001,wide_002,wide_003,wide_004,wide_005,...,wide_250,wide_251,wide_252,wide_253,wide_254,wide_255,wide_256,wide_257,wide_258,wide_259
0,m0,ok,2024-01-01 08:00:00+00:00,2024-01-01 08:10:00+00:00,valor_0,valor_1,valor_2,valor_3,valor_4,valor_5,...,valor_250,valor_251,valor_252,valor_253,valor_254,valor_255,valor_256,valor_257,valor_258,valor_259


[Issue(level='warning', code='IMP.COLUMNS.WIDE_TABLE', message='El TripDataset resultante tiene 264 columnas, superando el soft cap 256; se permite continuar, pero la tabla es ancha y puede generar fricción operativa.', field=None, source_field=None, row_count=None, details={'n_columns': 264, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': [], 'extra_fields_kept_total': 0, 'action': 'allow_with_warning'}),
 Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_e0affd069c3f4f4a8c31bb0232720112'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_e0affd069c3f4f4a8c31bb0232720112', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]

Columnas finales: 264
Codes de cap detectados: ['IMP.COLUMNS.WIDE_TABLE']
Detalles del warning: {'n_columns': 264, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': [], 'extra_fields_kept_total': 0, 'action': 'allow_with_warning'}
extra_fields_kept: []
OK: soft cap detectado correctamente con columnas definidas solo en el schema.


### Test 3 - Import con hard cap excedido usando solo campos extra

En esta prueba se construye un `TripSchema` pequeño y se fuerza el exceso de columnas mediante campos extra que no forman parte del schema. El objetivo es verificar que el import sea rechazado con la excepción correspondiente y que el code/issue gatillante sea `IMP.COLUMNS.HARD_CAP_EXCEEDED`.

In [8]:
n_extra_fields_hard = 1022

schema_hard_extras_only = TripSchema(
    version="1.0-test",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "core_col": FieldSpec(name="core_col", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
    },
    required=["movement_id", "core_col"],
)

row_hard_extras = {
    "movement_id": "m0",
    "core_col": "ok",
    "origin_time_utc": "2024-01-01T08:00:00Z",
    "destination_time_utc": "2024-01-01T08:10:00Z",
}

for i in range(n_extra_fields_hard):
    row_hard_extras[f"extra_{i:04d}"] = f"valor_{i}"

df_hard_extras = pd.DataFrame([row_hard_extras])

print("Inspección de dataframe WIDE que se quiere importar")
display(df_hard_extras)

try:
    _trips_hard, _report_hard = import_trips_from_dataframe(
        df=df_hard_extras,
        schema=schema_hard_extras_only,
    )
    raise AssertionError("Se esperaba una excepción por hard cap, pero el import continuó.")
except PylondrinaImportError as exc:
    print("Tipo de excepción:", type(exc).__name__)
    print("Code de la excepción:", exc.code)
    print("Code del issue gatillante:", exc.issue.code if exc.issue is not None else None)
    print("Details:", exc.details)
    print("Issues acumulados:", [issue.code for issue in (exc.issues or [])])

    assert exc.code == "IMP.COLUMNS.HARD_CAP_EXCEEDED"
    assert exc.issue is not None
    assert exc.issue.code == "IMP.COLUMNS.HARD_CAP_EXCEEDED"
    assert exc.details["n_columns"] > HARD_CAP
    assert exc.details["extra_fields_kept_total"] == n_extra_fields_hard
    assert [issue.code for issue in (exc.issues or [])] == ["IMP.COLUMNS.HARD_CAP_EXCEEDED"]

    print("OK: hard cap detectado correctamente con columnas extra fuera del schema.")

Inspección de dataframe WIDE que se quiere importar


,movement_id,core_col,origin_time_utc,destination_time_utc,extra_0000,extra_0001,extra_0002,extra_0003,extra_0004,extra_0005,...,extra_1012,extra_1013,extra_1014,extra_1015,extra_1016,extra_1017,extra_1018,extra_1019,extra_1020,extra_1021
0,m0,ok,2024-01-01T08:00:00Z,2024-01-01T08:10:00Z,valor_0,valor_1,valor_2,valor_3,valor_4,valor_5,...,valor_1012,valor_1013,valor_1014,valor_1015,valor_1016,valor_1017,valor_1018,valor_1019,valor_1020,valor_1021


Tipo de excepción: ImportError
Code de la excepción: IMP.COLUMNS.HARD_CAP_EXCEEDED
Code del issue gatillante: IMP.COLUMNS.HARD_CAP_EXCEEDED
Details: {'n_columns': 1026, 'soft_cap': 256, 'hard_cap': 1024, 'extra_fields_kept_sample': ['extra_0000', 'extra_0001', 'extra_0002', 'extra_0003', 'extra_0004', 'extra_0005', 'extra_0006', 'extra_0007', 'extra_0008', 'extra_0009'], 'extra_fields_kept_total': 1022, 'action': 'abort'}
Issues acumulados: ['IMP.COLUMNS.HARD_CAP_EXCEEDED']
OK: hard cap detectado correctamente con columnas extra fuera del schema.
